# 工具的应用案例


## 1.使用args_schema

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.tools.convert import tool

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
)

class WeatherSchema(BaseModel):
    city : str = Field(default = "北京", description = "具体的城市名称")
    if_forecast: bool = Field(default=False, description="是否包含明天天气")

# 定义工具
@tool("get_weather_and_forecast", description = "查询当日天气，可以包含明天的天气预报", args_schema = WeatherSchema)
def get_weather(city: str, if_forecast:bool):
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res
# rprint(convert_to_openai_tool(get_weather))


In [ ]:
# 将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 维护一个消息列表
messages = [HumanMessage("今天杭州的天气如何？明天呢?")]

# 调用模型,得到响应:AIMessage
res = model_with_tools.invoke(messages)

messages.append(res)

# 获取响应中的tool_calls字段信息
tool_calls = res.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        # 调用工具(因为大模型不能直接调用工具，所有此时我们主动让这个工具调用执行)

        # 调用完返回ToolMessage的实例
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 调用模型,得到AIMessage
final_res = model.invoke(messages)

# 添加到消息列表中
messages.append(final_res)

# 遍历消息列表
for msg in messages:
    msg.pretty_print()

## 2.撰写docstring

In [10]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
)


# 定义工具
@tool("get_weather_and_forecast", parse_docstring=True)
def get_weather(city: str = "北京", if_forecast:bool = False):
    """
    查询当日天气，可以包含明天的天气预报

    Args:
        city : 城市名称
        if_forecast: 是否包含明天的天气
    """
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res

# rprint(convert_to_openai_tool(get_weather))

# 将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 维护一个消息列表
messages = [HumanMessage("今天杭州的天气如何？明天呢?")]

# 调用模型,得到响应:AIMessage
res = model_with_tools.invoke(messages)

messages.append(res)

# 获取响应中的tool_calls字段信息
tool_calls = res.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        # 调用工具(因为大模型不能直接调用工具，所有此时我们主动让这个工具调用执行)

        # 调用完返回ToolMessage的实例
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 调用模型,得到AIMessage
final_res = model.invoke(messages)

# 添加到消息列表中
messages.append(final_res)

# 遍历消息列表
for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

今天杭州的天气如何？明天呢?
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_00_dUUl8YOC9souFew7ZGDT9813)
 Call ID: call_00_dUUl8YOC9souFew7ZGDT9813
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州今天天气不错
明天下雨
================================== Ai Message ==================================

杭州今天天气不错；明天会下雨，出门记得带伞。
